# 04a — NDVI Feature Engineering

Replicates v1's NDVI pipeline at 100 m² tile resolution via `applyInPandas`.

Per tile: Hampel outlier filter → 7-day rolling smooth → daily time interpolation per year → weekly OLS slope + mean + std.

Output: `data/spark/ndvi_weekly/` — one row per (tile_id, year, week).

In [1]:
import os, glob

os.environ.setdefault('JAVA_HOME', '/usr/lib/jvm/java-17-openjdk-amd64')
os.environ['PYSPARK_PYTHON']        = '/home/simonhans/anaconda3/envs/GrapeExpectationsML/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/home/simonhans/anaconda3/envs/GrapeExpectationsML/bin/python'

NDVI_DIR = '../data/ndvi/tiles_100m2'
OUT_DIR  = '../data/spark/ndvi_weekly'
os.makedirs(OUT_DIR, exist_ok=True)

In [2]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName('GrapeExpectations-v2-NDVIFeatures')
    .master('local[*]')
    .config('spark.driver.memory', '16g')
    .config('spark.driver.maxResultSize', '4g')
    .config('spark.sql.shuffle.partitions', '400')
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
    .getOrCreate())

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/19 22:54:22 WARN Utils: Your hostname, office, resolves to a loopback address: 127.0.1.1; using 192.168.86.43 instead (on interface wlo1)
26/05/19 22:54:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/19 22:54:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/19 22:54:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/19 22:54:23 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark version: 4.1.1


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, IntegerType, DoubleType, StringType

raw_schema = StructType([
    StructField('tile_id', LongType(),   True),
    StructField('date',    StringType(), True),
    StructField('ndvi',    DoubleType(), True),
])

csv_files = glob.glob(os.path.join(NDVI_DIR, 'ndvi_100m2_*.csv'))
print(f'Found {len(csv_files)} year CSVs')

raw = (spark.read
    .option('header', True)
    .schema(raw_schema)
    .csv(csv_files))

print(f'Raw observations: {raw.count():,}')
raw.show(3)

Found 10 year CSVs
Raw observations: 16,291,066
+-------+----------+----------+
|tile_id|      date|      ndvi|
+-------+----------+----------+
|      0|2021-02-28|0.31760436|
|      1|2021-02-28|0.37745973|
|      2|2021-02-28|0.40140024|
+-------+----------+----------+
only showing top 3 rows


In [4]:
OUT_SCHEMA = 'tile_id long, year int, week int, ndvi_smooth_mean double, ndvi_smooth_std double, ndvi_smooth_slope double'

OUT_COLS = ['tile_id', 'year', 'week', 'ndvi_smooth_mean', 'ndvi_smooth_std', 'ndvi_smooth_slope']


def compute_ndvi_features(df):
    """
    Input: all raw NDVI observations for one tile_id (all years).
    Output: weekly (mean, std, OLS slope) of the smoothed+interpolated NDVI curve.
    """
    import numpy as np
    import pandas as pd

    empty = pd.DataFrame(columns=OUT_COLS).astype({
        'tile_id': 'int64', 'year': 'int32', 'week': 'int32',
        'ndvi_smooth_mean': 'float64', 'ndvi_smooth_std': 'float64', 'ndvi_smooth_slope': 'float64',
    })

    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').dropna(subset=['ndvi']).reset_index(drop=True)

    if len(df) < 3:
        return empty

    tile_id = int(df['tile_id'].iloc[0])

    # ── Hampel outlier filter (all years together) ──────────────────────────
    s = df['ndvi'].copy()
    window, k, n_sig = 7, 1.4826, 2
    rolling_med = s.rolling(2 * window + 1, center=True, min_periods=1).median()
    rolling_mad = s.rolling(2 * window + 1, center=True, min_periods=1).apply(
        lambda x: np.median(np.abs(x - np.median(x))), raw=True
    ).replace(0, 1e-6)
    s[np.abs(s - rolling_med) > n_sig * k * rolling_mad] = np.nan
    df['ndvi'] = s.values

    # ── 7-day rolling smooth ────────────────────────────────────────────────
    df['ndvi_smooth'] = df['ndvi'].rolling(7, center=True, min_periods=1).mean()

    # ── Fill remaining NaNs ─────────────────────────────────────────────────
    df['ndvi'] = df['ndvi'].fillna(df['ndvi_smooth'])
    df['ndvi'] = df['ndvi'].fillna(df['ndvi'].median())
    df['ndvi_smooth'] = df['ndvi_smooth'].fillna(df['ndvi_smooth'].median())

    results = []
    for year, g in df.groupby(df['date'].dt.year):
        year = int(year)

        # Daily interpolation within year
        full_idx = pd.date_range(f'{year}-01-01', f'{year}-12-31', freq='D')
        g_daily = (
            g.set_index('date')[['ndvi_smooth']]
             .groupby(level=0).mean()
             .reindex(full_idx)
             .interpolate(method='time', limit_direction='both')
        )
        if g_daily['ndvi_smooth'].isna().all():
            continue

        g_daily = g_daily.reset_index(names='date')
        g_daily['week'] = g_daily['date'].dt.isocalendar().week.astype(int)
        g_daily['ord']  = g_daily['date'].map(pd.Timestamp.toordinal)

        # Weekly OLS slope + mean + std
        for week, wg in g_daily.groupby('week'):
            vals = wg['ndvi_smooth'].values
            x    = wg['ord'].values.astype(float)
            mean_v  = float(np.mean(vals))
            std_v   = float(np.std(vals, ddof=0))
            xv = np.var(x)
            slope_v = float(np.cov(x, vals, ddof=0)[0, 1] / xv) if xv > 0 else 0.0
            results.append((tile_id, year, int(week), mean_v, std_v, slope_v))

    if not results:
        return empty

    out = pd.DataFrame(results, columns=OUT_COLS)
    out['year'] = out['year'].astype('int32')
    out['week'] = out['week'].astype('int32')
    return out

In [5]:
# Repartition so each partition holds a manageable set of tiles
n_tiles = raw.select('tile_id').distinct().count()
print(f'Tiles: {n_tiles:,}')

# ~100 tiles per partition keeps each pandas group small
n_parts = max(200, n_tiles // 100)
raw_parts = raw.repartition(n_parts, 'tile_id')

print(f'Running applyInPandas across {n_parts} partitions…')
ndvi_weekly = raw_parts.groupby('tile_id').applyInPandas(compute_ndvi_features, schema=OUT_SCHEMA)

ndvi_weekly.write.mode('overwrite').parquet(OUT_DIR)
print(f'Saved → {OUT_DIR}')

Tiles: 32,978
Running applyInPandas across 329 partitions…


Saved → ../data/spark/ndvi_weekly


In [6]:
check = spark.read.parquet(OUT_DIR)
print(f'Rows: {check.count():,}')
print('Nulls:')
check.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in check.columns]).show()
check.show(5)

Rows: 17,247,494
Nulls:
+-------+----+----+----------------+---------------+-----------------+
|tile_id|year|week|ndvi_smooth_mean|ndvi_smooth_std|ndvi_smooth_slope|
+-------+----+----+----------------+---------------+-----------------+
|      0|   0|   0|               0|              0|                0|
+-------+----+----+----------------+---------------+-----------------+

+-------+----+----+----------------+---------------+-----------------+
|tile_id|year|week|ndvi_smooth_mean|ndvi_smooth_std|ndvi_smooth_slope|
+-------+----+----+----------------+---------------+-----------------+
|    150|2016|   1|      0.30410871|            0.0|              0.0|
|    150|2016|   2|      0.30410871|            0.0|              0.0|
|    150|2016|   3|      0.30410871|            0.0|              0.0|
|    150|2016|   4|      0.30410871|            0.0|              0.0|
|    150|2016|   5|      0.30410871|            0.0|              0.0|
+-------+----+----+----------------+---------------+